## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the LOINC Clinical Terminology Mapper model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *LOINC Clinical Terminology Mapper* offer in the [Azure Marketplace](https://azuremarketplace.microsoft.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-authenticate-online-endpoint) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## LOINC Clinical Terminology Mapper

- **Model**: [loinc_vdb_resolver](https://nlp.johnsnowlabs.com/2024/10/08/loinc_resolver_pipeline_en.html)

- **Model Description**: This pretrained pipeline extracts clinical entities from clinical text and maps them to their corresponding Logical Observation Identifiers Names and Codes (LOINC) codes.

In [1]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**:
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }
   ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
   {
       "text": "Single text document"
   }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.


### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import pandas as pd
import requests
import json

# Set display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record) if content_type == "application/json" else record,

    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return response_body
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.


In [3]:
docs = [
    "The patient is a 22-year-old female with a history of obesity. She has a Body mass index (BMI) of 33.5 kg/m2, aspartate aminotransferase 64, and alanine aminotransferase 126.",
    "Final diagnosis:  : DNA was extracted from the peripheral blood specimen and a polymerase chain reaction (PCR)-based assay performed that is designed to detect the presence of two separate mutations in the FLT3 gene: (1) within a susceptible region that includes coding sequence for the intracellular juxtamembrane domain and (2)   Negative. Neither expansion of the region susceptible to nor changes consistent with mutation of the codon for ASP835 were identified.",
]

sample_text = """A 65-year-old woman presents to the office with generalized fatigue for the last 4 months. She used to walk 1 mile each evening but now gets tired after 1-2 blocks. She has a history of Crohn disease and hypertension for which she receives appropriate medications. She is married and lives with her husband. She eats a balanced diet that includes chicken, fish, pork, fruits, and vegetables. She rarely drinks alcohol and denies tobacco use. Her vital signs are within normal limits. A physical examination is unremarkable. Laboratory studies show the following: Hemoglobin: 9.8 g/dL, Hematocrit: 32%, Mean Corpuscular Volume: 110 μm3"""

### JSON

#### Example 1: single text document

In [4]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
pd.DataFrame(response_json["predictions"])

,begin,end,ner_chunk,ner_label,ner_confidence,concept_code,resolution,score,all_codes,domain_id,all_resolutions,all_score
0,442,456,Her vital signs,Test,0.78679997,8716-3,vital signs,0.850316,"[8716-3, LP139953-6, LP30605-7, 8716-3, LP75862-0]","[Observation, Observation, Measurement, , Observation]","[vital signs, vital signs, vital signs, vital signs note, vital signs - acute]","[0.850316047668457, 0.850316047668457, 0.8503159284591675, 0.8003305196762085, 0.7810258865356445]"
1,484,505,A physical examination,Test,0.91120005,LP7801-6,physical exam,0.909418,"[LP7801-6, LP94385-9, 29271-4, LP74050-3, 100223-7]","[Measurement, Observation, Observation, Observation, ]","[physical exam, physical examination by body areas, eye physical examination, eye physical examination, examination]","[0.909417986869812, 0.8629758358001709, 0.8507089018821716, 0.8507089018821716, 0.840781033039093]"
2,524,541,Laboratory studies,Test,0.7098,12253-1,bs; chemistry.challenge testing; endocrine; endocrinology; gh; growth hormone; hgh; human growth hormone; level; mass concentration; pl; plasma; plsm; point in time; qnt; quan; quant; quantitative; random; serp; serpl; serplas; serum; serum or plasma; somatotropic hormone; sr; sth,0.684642,"[12253-1, 10872-0, 50667-5, 97107-7, 1547-9]","[, , , , ]","[bs; chemistry.challenge testing; endocrine; endocrinology; gh; growth hormone; hgh; human growth hormone; level; mass concentration; pl; plasma; plsm; point in time; qnt; quan; quant; quantitative; random; serp; serpl; serplas; serum; serum or plasma; somatotropic hormone; sr; sth, blood; bs; chemistry.challenge testing; d-xylose; level; mass concentration; point in time; qnt; quan; quant; quantitative; random; wb; whole blood; xylose absorption test, chemistry.challenge testing; endocrine; endocrinology; glu; glu tol; gluc; glucose tolerance; glucoseur; gtt; impression; impression/interpretation of study; impressions; interp; interpretation; narrative; pl; plasma; plsm; point in time; random; report; serp; serpl; serplas; serum; serum or plasma; sr, chemistry; eia; elfa; elisa; endocrine; endocrinology; enzyme immunoassay; fac; fact; fgf-19; hepatology; iaa; level; liver; mass concentration; meia; pl; plasma; plsm; point in time; qnt; quan; quant; quantitative; random; serp; serpl; serplas; serum; serum or plasma; sr; suds, bs; chemistry.challenge testing; endocrine; endocrinology; glu; glu tol; gluc; glucose tolerance; glucoseur; gtt; level; mass concentration; pl; plasma; plsm; point in time; qnt; quan; quant; quantitative; random; serp; serpl; serplas; serum; serum or plasma; sr]","[0.6846417784690857, 0.6821582913398743, 0.6818782091140747, 0.6808785796165466, 0.6796897053718567]"
3,563,572,Hemoglobin,Test,0.9994,LP14449-0,hemoglobin,1.000000,"[LP14449-0, 10346-5, 109592-6, LP49432-5, LP43135-0]","[Observation, , , Measurement, Measurement]","[hemoglobin, haemoglobin, hemiglobin, hemoglobin | blood venous, hemoglobin | bld-ser-plas]","[0.9999997019767761, 0.9589887261390686, 0.9433582425117493, 0.8918702602386475, 0.8897995352745056]"
4,579,582,g/dL,Test,0.6455,18286-5,dl,0.829473,"[18286-5, 76650-1, 43740-0, 2367-1, LA19932-5]","[, , , , Meas Value]","[dl, gd, g/i, gld, g/g (wild type)]","[0.8294733762741089, 0.809005856513977, 0.8026497960090637, 0.7987962961196899, 0.7837278246879578]"
5,585,594,Hematocrit,Test,0.9983,LP15101-6,hematocrit,0.999999,"[LP15101-6, 32354-3, LP45040-0, LP52290-1, LP106437-9]","[Observation, , Measurement, Measurement, Measurement]","[hematocrit, hematocrit, blood, hematocrit | bld-ser-plas, hematocrit | blood venous, hematocrit | stem cell product]","[0.999998927116394, 0.9471284747123718, 0.9460045695304871, 0.94156813621521, 0.9407790899276733]"
6,602,624,Mean Corpuscular Volume,Test,0.7142666,51641-9,average; avg; entitic volume; erythrocytes; hematology/cell counts; mscv; point in time; qnt; quan; quant; quantitative; random; red blood cells; red blood corpusles; vol,0.739193,"[51641-9, 48706-6, 68402-7, 68902-6, 9

#### Example 2: array of text documents

In [5]:
input_json_data = {"text": docs}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': [[{'begin': 90,
    'end': 93,
    'ner_chunk': 'BMI)',
    'ner_label': 'Test',
    'ner_confidence': '0.698',
    'concept_code': '39156-5',
    'resolution': 'bmi',
    'score': 0.9296669960021973,
    'all_codes': ['39156-5', 'LP35925-4', '89270-3', '107102-6', '59574-4'],
    'domain_id': ['', 'Observation', '', '', ''],
    'all_resolutions': ['bmi',
     'body mass index (bmi)',
     'bmi est',
     'bmi formula',
     'bmi prctl'],
    'all_score': [0.9296669960021973,
     0.8603405952453613,
     0.8320397734642029,
     0.8136729001998901,
     0.8040130734443665]},
   {'begin': 110,
    'end': 135,
    'ner_chunk': 'aspartate aminotransferase',
    'ner_label': 'Test',
    'ner_confidence': '0.8905',
    'concept_code': 'LP15426-7',
    'resolution': 'aspartate aminotransferase',
    'score': 0.9999998211860657,
    'all_codes': ['LP15426-7',
     '100739-2',
     'LP307348-5',
     'LP307326-1',
     '100739-2'],
    'domain_id': ['Observation', '', 'Observ

### JSON Lines

In [6]:
def create_jsonl(records):
    if isinstance(records, str):
        records = [records]
    json_records = [{"text": text} for text in records]
    json_lines = "\n".join(json.dumps(record) for record in json_records)
    return json_lines

In [7]:
input_jsonl_data = create_jsonl(docs)
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": [{"begin": 90, "end": 93, "ner_chunk": "BMI)", "ner_label": "Test", "ner_confidence": "0.698", "concept_code": "39156-5", "resolution": "bmi", "score": 0.9296669960021973, "all_codes": ["39156-5", "LP35925-4", "89270-3", "107102-6", "59574-4"], "domain_id": ["", "Observation", "", "", ""], "all_resolutions": ["bmi", "body mass index (bmi)", "bmi est", "bmi formula", "bmi prctl"], "all_score": [0.9296669960021973, 0.8603405952453613, 0.8320397734642029, 0.8136729001998901, 0.8040130734443665]}, {"begin": 110, "end": 135, "ner_chunk": "aspartate aminotransferase", "ner_label": "Test", "ner_confidence": "0.8905", "concept_code": "LP15426-7", "resolution": "aspartate aminotransferase", "score": 0.9999998211860657, "all_codes": ["LP15426-7", "100739-2", "LP307348-5", "LP307326-1", "100739-2"], "domain_id": ["Observation", "", "Observation", "Observation", ""], "all_resolutions": ["aspartate aminotransferase", "aspartate amino transferase", "aspartate aminotransferase/alanine